# Fase 5 - Clustering Geografico: Centros de Actividad de NYC

**Pregunta de negocio:** ¿Cuales son los centros de actividad de NYC y como se diferencian?

## Objetivos de este notebook

1. Consultar features agregadas por zona TLC (location_id) desde BigQuery
2. Aplicar DBSCAN y K-Means sobre features de zona para descubrir clusters de actividad
3. Comparar DBSCAN vs K-Means en el espacio de features por zona
4. Visualizar clusters en mapas interactivos con Folium usando centroides de zona
5. Aplicar t-SNE y PCA para reduccion dimensional del espacio de features
6. Identificar hotspots de recogida (Manhattan, JFK, LaGuardia, etc.)

## Enfoque: Clustering por features de zona

La tabla `tlc_yellow_trips_2015` no contiene coordenadas GPS, sino `pickup_location_id`
y `dropoff_location_id` (TLC Zone IDs, 1-263). En lugar de clustering espacial sobre
coordenadas, **agregamos features por zona** (cantidad de viajes, tarifa promedio,
propina promedio, distancia promedio) y aplicamos clustering sobre ese espacio de features.

## ¿Por que DBSCAN es util incluso sin datos GPS?

A diferencia de K-Means, **DBSCAN** (Density-Based Spatial Clustering of Applications with Noise):
- No requiere especificar el numero de clusters a priori
- Detecta clusters de **forma arbitraria** (no solo esfericos) en el espacio de features
- Identifica automaticamente **puntos de ruido** (zonas atipicas)
- Es ideal para descubrir agrupaciones naturales en features multidimensionales

## 1. Configuracion e importaciones

In [ ]:
# Conexion a BigQuery
import sys
sys.path.insert(0, '../../../src')
from bigquery.bq_helper import BigQueryHelper
bq = BigQueryHelper()

In [ ]:
# Librerias de analisis y visualizacion
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import DBSCAN, KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import silhouette_score
import folium
from folium.plugins import HeatMap
import warnings
warnings.filterwarnings('ignore')

# Configuracion de estilo
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)

RANDOM_STATE = 42

## 2. Carga de features agregadas por zona TLC

Agregamos metricas clave por cada `pickup_location_id` (zona TLC):
- **trip_count:** numero total de viajes originados en la zona
- **avg_fare:** tarifa promedio
- **avg_tip:** propina promedio
- **avg_tip_pct:** porcentaje de propina promedio
- **avg_distance:** distancia promedio del viaje
- **avg_duration_min:** duracion promedio en minutos

Cada zona TLC se convierte en una "observacion" con estas features.

In [ ]:
query = """
SELECT
    pickup_location_id AS location_id,
    COUNT(*) AS trip_count,
    ROUND(AVG(fare_amount), 2) AS avg_fare,
    ROUND(AVG(tip_amount), 2) AS avg_tip,
    ROUND(AVG(CASE WHEN fare_amount > 0 AND payment_type = '1'
              THEN tip_amount / fare_amount * 100 ELSE NULL END), 2) AS avg_tip_pct,
    ROUND(AVG(trip_distance), 2) AS avg_distance,
    ROUND(AVG(TIMESTAMP_DIFF(dropoff_datetime, pickup_datetime, SECOND)) / 60.0, 2) AS avg_duration_min,
    EXTRACT(HOUR FROM pickup_datetime) AS hour
FROM
    `bigquery-public-data.new_york_taxi_trips.tlc_yellow_trips_2015`
WHERE
    fare_amount > 0 AND fare_amount < 500
    AND trip_distance > 0 AND trip_distance < 100
    AND pickup_location_id IS NOT NULL
    AND TIMESTAMP_DIFF(dropoff_datetime, pickup_datetime, SECOND) > 60
    AND TIMESTAMP_DIFF(dropoff_datetime, pickup_datetime, SECOND) < 7200
    AND pickup_datetime >= '2015-01-01'
    AND pickup_datetime < '2016-01-01'
GROUP BY
    location_id, hour
"""

df_raw = bq.query_to_df(query)
print(f"Registros cargados (zona x hora): {len(df_raw):,}")
print(f"Zonas unicas: {df_raw['location_id'].nunique()}")

# Agregar a nivel de zona (sin hora) para clustering
df = df_raw.groupby('location_id').agg(
    trip_count=('trip_count', 'sum'),
    avg_fare=('avg_fare', 'mean'),
    avg_tip=('avg_tip', 'mean'),
    avg_tip_pct=('avg_tip_pct', 'mean'),
    avg_distance=('avg_distance', 'mean'),
    avg_duration_min=('avg_duration_min', 'mean')
).reset_index()

# Filtrar zonas con muy pocos viajes (ruido estadistico)
min_trips = 1000
df = df[df['trip_count'] >= min_trips].copy()
print(f"\nZonas con >= {min_trips:,} viajes: {len(df)}")
print(f"\nEstadisticas descriptivas:")
df.describe().round(2)

In [ ]:
# Limpiar datos nulos
df_clean = df.dropna().copy()
print(f"Zonas despues de limpieza: {len(df_clean):,}")
print(f"Zonas eliminadas: {len(df) - len(df_clean):,}")

## 3. DBSCAN en features de zona

### Parametros clave de DBSCAN:
- **eps:** radio maximo de vecindad en el espacio de features escalado
- **min_samples:** numero minimo de zonas para formar un cluster denso

### ¿Por que DBSCAN es mejor que K-Means para descubrir patrones?

| Caracteristica | K-Means | DBSCAN |
|:---|:---|:---|
| Forma de clusters | Solo esfericos | Formas arbitrarias |
| Numero de clusters | Hay que especificarlo | Se descubre automaticamente |
| Manejo de ruido | No (asigna todo) | Si (etiqueta -1) |
| Zonas atipicas | Se fuerzan a un cluster | Se identifican como ruido |

In [ ]:
# Preparar features para clustering
feature_cols = ['trip_count', 'avg_fare', 'avg_tip_pct', 'avg_distance', 'avg_duration_min']
X = df_clean[feature_cols].values

# Escalar features (fundamental para DBSCAN y K-Means)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Aplicar DBSCAN
# eps y min_samples ajustados para features escaladas
dbscan = DBSCAN(eps=1.2, min_samples=3, n_jobs=-1)
db_labels = dbscan.fit_predict(X_scaled)

df_clean['db_cluster'] = db_labels

# Resultados
n_clusters = len(set(db_labels)) - (1 if -1 in db_labels else 0)
n_noise = (db_labels == -1).sum()

print(f"Resultados de DBSCAN (eps=1.2, min_samples=3):")
print(f"  Clusters encontrados: {n_clusters}")
print(f"  Zonas de ruido:       {n_noise} ({n_noise/len(db_labels)*100:.1f}%)")
print(f"  Zonas en clusters:    {(db_labels != -1).sum()} ({(db_labels != -1).sum()/len(db_labels)*100:.1f}%)")

# Clusters por tamanio
cluster_sizes = pd.Series(db_labels[db_labels != -1]).value_counts()
print(f"\nTamanio de clusters:")
for cl, size in cluster_sizes.items():
    print(f"  Cluster {cl}: {size} zonas")

## 4. Visualizacion de clusters DBSCAN

Visualizamos los clusters DBSCAN en el espacio de features usando las dos variables
mas informativas: cantidad de viajes y tarifa promedio. Las zonas de ruido (label=-1)
se muestran en gris.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Scatter: trip_count vs avg_fare, coloreado por cluster
ax = axes[0]
noise_mask = df_clean['db_cluster'] == -1
ax.scatter(df_clean.loc[noise_mask, 'trip_count'],
           df_clean.loc[noise_mask, 'avg_fare'],
           c='lightgray', s=60, alpha=0.6, label='Ruido', edgecolors='gray')

cluster_mask = df_clean['db_cluster'] != -1
scatter = ax.scatter(df_clean.loc[cluster_mask, 'trip_count'],
                     df_clean.loc[cluster_mask, 'avg_fare'],
                     c=df_clean.loc[cluster_mask, 'db_cluster'],
                     cmap='tab10', s=80, alpha=0.7, edgecolors='white')

ax.set_xlabel('Cantidad de viajes (trip_count)')
ax.set_ylabel('Tarifa promedio (USD)')
ax.set_title(f'Clusters DBSCAN: Viajes vs Tarifa ({n_clusters} clusters)', fontweight='bold')
plt.colorbar(scatter, ax=ax, label='Cluster ID', shrink=0.7)

# Scatter: avg_distance vs avg_tip_pct, coloreado por cluster
ax = axes[1]
ax.scatter(df_clean.loc[noise_mask, 'avg_distance'],
           df_clean.loc[noise_mask, 'avg_tip_pct'],
           c='lightgray', s=60, alpha=0.6, label='Ruido', edgecolors='gray')

scatter2 = ax.scatter(df_clean.loc[cluster_mask, 'avg_distance'],
                      df_clean.loc[cluster_mask, 'avg_tip_pct'],
                      c=df_clean.loc[cluster_mask, 'db_cluster'],
                      cmap='tab10', s=80, alpha=0.7, edgecolors='white')

ax.set_xlabel('Distancia promedio (millas)')
ax.set_ylabel('Propina promedio (%)')
ax.set_title(f'Clusters DBSCAN: Distancia vs Propina', fontweight='bold')
plt.colorbar(scatter2, ax=ax, label='Cluster ID', shrink=0.7)

plt.suptitle('Clustering DBSCAN sobre Features Agregadas por Zona TLC', fontweight='bold', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

## 5. Mapa interactivo con Folium

Visualizamos los clusters en un mapa de NYC usando Folium. Como no tenemos coordenadas
GPS por viaje, asignamos a cada zona su borough/area y usamos centroides aproximados.
El tamanio del marcador refleja la cantidad de viajes de la zona.

In [ ]:
# Definir zonas conocidas y centroides
MANHATTAN_ZONES = {'4','12','13','24','41','42','43','45','48','50','68','74','75','79','87','88','90','100','107','113','114','116','125','127','128','137','140','141','142','143','144','148','151','152','153','158','161','162','163','164','166','170','186','194','202','209','211','224','229','230','231','232','233','234','236','237','238','239','243','244','246','249','261','262','263'}
BROOKLYN_ZONES = {'11','14','17','21','22','25','26','29','33','34','35','36','37','39','40','49','52','54','55','61','62','63','65','66','67','69','71','72','76','77','80','85','89','91','97','106','108','111','112','123','133','149','150','154','155','165','177','178','181','188','189','190','195','210','217','222','225','227','228','255','256','257'}

ZONE_CENTROIDS = {
    'Manhattan': (40.7580, -73.9855),
    'Brooklyn': (40.6782, -73.9442),
    'Queens/Otro': (40.7282, -73.7949),
    'JFK': (40.6413, -73.7781),
    'LaGuardia': (40.7769, -73.8740),
}

def classify_zone(loc_id):
    """Clasifica una zona TLC en su borough/area."""
    if loc_id in MANHATTAN_ZONES:
        return 'Manhattan'
    elif loc_id in BROOKLYN_ZONES:
        return 'Brooklyn'
    elif loc_id == '132':
        return 'JFK'
    elif loc_id == '138':
        return 'LaGuardia'
    else:
        return 'Queens/Otro'

df_clean['borough'] = df_clean['location_id'].apply(classify_zone)
df_clean['lat'] = df_clean['borough'].map(lambda b: ZONE_CENTROIDS.get(b, (40.7128, -74.0060))[0])
df_clean['lon'] = df_clean['borough'].map(lambda b: ZONE_CENTROIDS.get(b, (40.7128, -74.0060))[1])

# Agregar pequeno jitter para que los puntos del mismo borough no se superpongan
np.random.seed(RANDOM_STATE)
df_clean['lat'] = df_clean['lat'] + np.random.normal(0, 0.008, len(df_clean))
df_clean['lon'] = df_clean['lon'] + np.random.normal(0, 0.008, len(df_clean))

# Crear mapa
center_lat, center_lon = 40.7380, -73.9200
m = folium.Map(location=[center_lat, center_lon], zoom_start=11, tiles='CartoDB positron')

# Colores para los clusters
cluster_colors = ['red', 'blue', 'green', 'purple', 'orange', 'darkred',
                  'cadetblue', 'darkgreen', 'darkpurple', 'pink']

# Plotear cada zona como un circulo
for _, row in df_clean.iterrows():
    cl = int(row['db_cluster'])
    if cl == -1:
        color = 'gray'
        label = 'Ruido'
    else:
        color = cluster_colors[cl % len(cluster_colors)]
        label = f'Cluster {cl}'
    
    radius = max(3, min(15, row['trip_count'] / df_clean['trip_count'].max() * 15))
    
    folium.CircleMarker(
        location=[row['lat'], row['lon']],
        radius=radius,
        color=color,
        fill=True,
        fill_opacity=0.6,
        popup=f"Zona {row['location_id']} ({row['borough']})<br>"
              f"{label}<br>"
              f"Viajes: {row['trip_count']:,.0f}<br>"
              f"Tarifa: ${row['avg_fare']:.2f}<br>"
              f"Propina: {row['avg_tip_pct']:.1f}%"
    ).add_to(m)

print("Mapa interactivo con clusters DBSCAN por zona TLC:")
m

## 6. K-Means en las mismas features de zona (para comparacion)

Aplicamos K-Means con el mismo numero de clusters que DBSCAN encontro
para comparar ambos algoritmos en el espacio de features de zona.

In [ ]:
# Usar el mismo numero de clusters que DBSCAN encontro
k_compare = max(n_clusters, 3)  # al menos 3 clusters para comparacion
print(f"Aplicando K-Means con k={k_compare} clusters para comparacion...")

kmeans_geo = KMeans(n_clusters=k_compare, random_state=RANDOM_STATE, n_init=10)
km_labels = kmeans_geo.fit_predict(X_scaled)

df_clean['km_cluster'] = km_labels

print(f"K-Means completado.")
print(f"Tamanio de clusters K-Means:")
for cl in range(k_compare):
    n = (km_labels == cl).sum()
    print(f"  Cluster {cl}: {n} zonas ({n/len(km_labels)*100:.1f}%)")

## 7. Comparacion visual: DBSCAN vs K-Means

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# DBSCAN
ax = axes[0]
noise = df_clean['db_cluster'] == -1
ax.scatter(df_clean.loc[noise, 'trip_count'],
           df_clean.loc[noise, 'avg_fare'],
           c='lightgray', s=60, alpha=0.5, label='Ruido', edgecolors='gray')
ax.scatter(df_clean.loc[~noise, 'trip_count'],
           df_clean.loc[~noise, 'avg_fare'],
           c=df_clean.loc[~noise, 'db_cluster'],
           cmap='tab10', s=80, alpha=0.7, edgecolors='white')
ax.set_title(f'DBSCAN ({n_clusters} clusters + ruido)', fontweight='bold', fontsize=14)
ax.set_xlabel('Cantidad de viajes')
ax.set_ylabel('Tarifa promedio (USD)')

# K-Means
ax = axes[1]
scatter_km = ax.scatter(df_clean['trip_count'],
                        df_clean['avg_fare'],
                        c=df_clean['km_cluster'],
                        cmap='tab10', s=80, alpha=0.7, edgecolors='white')

# Plotear centroides (en espacio original)
centers_original = scaler.inverse_transform(kmeans_geo.cluster_centers_)
ax.scatter(centers_original[:, 0], centers_original[:, 1],
           c='black', marker='X', s=200,
           edgecolors='white', linewidths=2, zorder=5, label='Centroides')
ax.set_title(f'K-Means ({k_compare} clusters)', fontweight='bold', fontsize=14)
ax.set_xlabel('Cantidad de viajes')
ax.set_ylabel('Tarifa promedio (USD)')
ax.legend()

plt.suptitle('Comparacion: DBSCAN vs K-Means en Features de Zona', fontweight='bold', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

# Metricas de comparacion
print("\nComparacion de metricas:")
print("=" * 50)

non_noise_mask = df_clean['db_cluster'] != -1
if non_noise_mask.sum() > 1 and len(set(db_labels[non_noise_mask])) > 1:
    sil_dbscan = silhouette_score(X_scaled[non_noise_mask], db_labels[non_noise_mask],
                                  random_state=RANDOM_STATE)
    print(f"DBSCAN Silhouette (sin ruido): {sil_dbscan:.4f}")

sil_kmeans = silhouette_score(X_scaled, km_labels, random_state=RANDOM_STATE)
print(f"K-Means Silhouette:            {sil_kmeans:.4f}")
print(f"\nDBSCAN zonas de ruido: {n_noise} ({n_noise/len(db_labels)*100:.1f}%)")
print(f"K-Means asigna TODAS las zonas a un cluster (sin deteccion de ruido)")

## 8. t-SNE: Visualizacion de features de zona por cluster

t-SNE (t-Distributed Stochastic Neighbor Embedding) es un algoritmo de reduccion
dimensional no lineal, excelente para visualizar la estructura local de los datos
en alta dimension.

Aplicamos t-SNE a las features de zona y coloreamos por cluster para evaluar
si los clusters son visualmente separables en un espacio reducido.

In [ ]:
# t-SNE sobre features de zona
# Solo zonas en clusters DBSCAN (no ruido) para visualizacion limpia
df_tsne = df_clean[df_clean['db_cluster'] != -1].copy()

# Si hay suficientes zonas, aplicar t-SNE
n_tsne = len(df_tsne)
print(f"Aplicando t-SNE a {n_tsne} zonas con {len(feature_cols)} features...")

X_tsne_input = scaler.transform(df_tsne[feature_cols])

# Ajustar perplexity segun el numero de muestras
perplexity = min(30, max(5, n_tsne // 3))
tsne = TSNE(n_components=2, random_state=RANDOM_STATE, perplexity=perplexity, n_iter=1000)
X_tsne = tsne.fit_transform(X_tsne_input)
print("t-SNE completado.")

# Visualizar
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Coloreado por cluster DBSCAN
ax = axes[0]
for cl in sorted(df_tsne['db_cluster'].unique()):
    mask = df_tsne['db_cluster'].values == cl
    ax.scatter(X_tsne[mask, 0], X_tsne[mask, 1],
               s=60, alpha=0.7, label=f'Cluster {cl} (n={mask.sum()})',
               edgecolors='white')

ax.set_xlabel('t-SNE Dimension 1')
ax.set_ylabel('t-SNE Dimension 2')
ax.set_title('t-SNE coloreado por Cluster DBSCAN', fontweight='bold')
ax.legend(title='Cluster DBSCAN', fontsize=9, markerscale=2)

# Coloreado por borough
ax = axes[1]
for borough in df_tsne['borough'].unique():
    mask = df_tsne['borough'].values == borough
    ax.scatter(X_tsne[mask, 0], X_tsne[mask, 1],
               s=60, alpha=0.7, label=f'{borough} (n={mask.sum()})',
               edgecolors='white')

ax.set_xlabel('t-SNE Dimension 1')
ax.set_ylabel('t-SNE Dimension 2')
ax.set_title('t-SNE coloreado por Borough', fontweight='bold')
ax.legend(title='Borough', fontsize=9, markerscale=2)

plt.suptitle('Reduccion Dimensional t-SNE de Features por Zona', fontweight='bold', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

print("\nInterpretacion:")
print("- Si los clusters se separan en t-SNE, las zonas dentro de cada cluster")
print("  comparten patrones de viaje similares.")
print("- Si boroughs y clusters coinciden, la geografia determina el patron de viaje.")

## 9. PCA: Varianza explicada, cargas y biplot

PCA nos ayuda a entender que combinaciones lineales de features capturan la mayor
varianza en los datos. A diferencia de t-SNE, PCA es interpretable: podemos examinar
las cargas (loadings) de cada componente.

In [ ]:
# PCA con todas las features de zona
pca = PCA(random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_scaled)

# Varianza explicada
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Varianza explicada por componente
axes[0].bar(range(1, len(feature_cols) + 1), pca.explained_variance_ratio_,
            color='steelblue', alpha=0.8, edgecolor='white')
axes[0].set_xlabel('Componente Principal')
axes[0].set_ylabel('Proporcion de Varianza Explicada')
axes[0].set_title('Varianza Explicada por Componente', fontweight='bold')
axes[0].set_xticks(range(1, len(feature_cols) + 1))

# Varianza acumulada
cumulative = np.cumsum(pca.explained_variance_ratio_)
axes[1].plot(range(1, len(feature_cols) + 1), cumulative, 'bo-', linewidth=2, markersize=8)
axes[1].axhline(y=0.9, color='red', linestyle='--', alpha=0.7, label='90% varianza')
axes[1].set_xlabel('Numero de Componentes')
axes[1].set_ylabel('Varianza Acumulada')
axes[1].set_title('Varianza Acumulada', fontweight='bold')
axes[1].set_xticks(range(1, len(feature_cols) + 1))
axes[1].set_ylim(0, 1.05)
axes[1].legend()

plt.suptitle('Analisis de Componentes Principales (PCA) - Features de Zona', fontweight='bold', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

# Tabla de varianza
print("\nVarianza explicada por componente:")
for i, (var, cum) in enumerate(zip(pca.explained_variance_ratio_, cumulative)):
    print(f"  PC{i+1}: {var:.2%} (acumulada: {cum:.2%})")

In [ ]:
# Cargas de los componentes (loadings)
loadings = pd.DataFrame(
    pca.components_.T,
    columns=[f'PC{i+1}' for i in range(len(feature_cols))],
    index=feature_cols
)

print("Cargas (loadings) de los componentes principales:")
display(loadings.round(3))

# Heatmap de loadings
fig, ax = plt.subplots(figsize=(10, 5))
sns.heatmap(loadings.iloc[:.astype(float), :3], annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            linewidths=1, ax=ax)
ax.set_title('Cargas de las 3 Primeras Componentes Principales', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Biplot: PCA scatter + vectores de features
fig, ax = plt.subplots(figsize=(12, 10))

# Scatter de zonas coloreadas por cluster DBSCAN
for cl in sorted(df_clean['db_cluster'].unique()):
    mask = df_clean['db_cluster'].values == cl
    label = f'Cluster {cl}' if cl != -1 else 'Ruido'
    color = 'lightgray' if cl == -1 else None
    alpha = 0.3 if cl == -1 else 0.7
    ax.scatter(X_pca[mask, 0], X_pca[mask, 1],
               s=60 if cl != -1 else 30, alpha=alpha, label=label,
               c=color, edgecolors='white' if cl != -1 else 'gray')

# Vectores de features (loadings)
scale = max(abs(X_pca[:, 0].max()), abs(X_pca[:, 1].max())) * 0.8
for i, feat in enumerate(feature_cols):
    ax.arrow(0, 0,
             loadings.iloc[i, 0] * scale,
             loadings.iloc[i, 1] * scale,
             head_width=0.08, head_length=0.04,
             fc='black', ec='black', linewidth=1.5)
    ax.text(loadings.iloc[i, 0] * scale * 1.15,
            loadings.iloc[i, 1] * scale * 1.15,
            feat, fontsize=10, fontweight='bold',
            bbox=dict(boxstyle='round,pad=0.2', facecolor='yellow', alpha=0.7))

ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} varianza)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} varianza)')
ax.set_title('Biplot PCA: Features de Zona por Cluster', fontweight='bold')
ax.legend(title='Cluster DBSCAN', fontsize=10, markerscale=2)
ax.axhline(y=0, color='gray', linestyle='-', alpha=0.3)
ax.axvline(x=0, color='gray', linestyle='-', alpha=0.3)

plt.tight_layout()
plt.show()

print("Interpretacion del biplot:")
print("- Las flechas indican la direccion y magnitud de cada feature en el espacio PCA")
print("- Flechas cercanas entre si = features correlacionadas")
print("- Flechas en direcciones opuestas = features inversamente correlacionadas")

## 10. Estadisticas por cluster de zona

Calculamos el promedio de tarifa, propina, distancia y cantidad de viajes para cada
cluster para entender las diferencias operativas entre grupos de zonas.

In [ ]:
# Estadisticas de los clusters
stats_cols = ['trip_count', 'avg_fare', 'avg_tip', 'avg_tip_pct', 'avg_distance', 'avg_duration_min']

cluster_summary = df_clean.groupby('db_cluster').agg(
    n_zonas=('location_id', 'count'),
    total_viajes=('trip_count', 'sum'),
    fare_mean=('avg_fare', 'mean'),
    tip_pct_mean=('avg_tip_pct', 'mean'),
    distance_mean=('avg_distance', 'mean'),
    duration_mean=('avg_duration_min', 'mean'),
).round(3)

cluster_summary = cluster_summary.sort_values('total_viajes', ascending=False)

# Agregar borough predominante
borough_mode = df_clean.groupby('db_cluster')['borough'].agg(lambda x: x.value_counts().index[0])
cluster_summary['borough_predominante'] = borough_mode

print("Estadisticas por cluster (DBSCAN):")
print("=" * 90)
display(cluster_summary)

In [ ]:
# Visualizacion de estadisticas por cluster
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Filtrar solo clusters reales (no ruido)
summary_no_noise = cluster_summary[cluster_summary.index != -1]
n_cl = len(summary_no_noise)
cl_ids = summary_no_noise.index

# Tarifa promedio
axes[0].barh(range(n_cl), summary_no_noise['fare_mean'],
             color='#2196F3', alpha=0.8)
axes[0].set_yticks(range(n_cl))
axes[0].set_yticklabels([f'Cluster {c}' for c in cl_ids])
axes[0].set_xlabel('Tarifa Promedio (USD)')
axes[0].set_title('Tarifa Promedio por Cluster', fontweight='bold')

# Propina promedio
axes[1].barh(range(n_cl), summary_no_noise['tip_pct_mean'],
             color='#4CAF50', alpha=0.8)
axes[1].set_yticks(range(n_cl))
axes[1].set_yticklabels([f'Cluster {c}' for c in cl_ids])
axes[1].set_xlabel('Propina Promedio (%)')
axes[1].set_title('Propina Promedio (%) por Cluster', fontweight='bold')

# Distancia promedio
axes[2].barh(range(n_cl), summary_no_noise['distance_mean'],
             color='#FF9800', alpha=0.8)
axes[2].set_yticks(range(n_cl))
axes[2].set_yticklabels([f'Cluster {c}' for c in cl_ids])
axes[2].set_xlabel('Distancia Promedio (millas)')
axes[2].set_title('Distancia Promedio por Cluster', fontweight='bold')

plt.suptitle('Comparacion de Clusters de Zonas TLC', fontweight='bold', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

## 11. Identificacion de hotspots conocidos

Asociamos los clusters con las zonas TLC conocidas (JFK, LaGuardia, zonas de Manhattan, etc.)
para interpretar los resultados del clustering.

In [ ]:
# Zonas conocidas con sus IDs TLC
known_zones = {
    '132': 'JFK Airport',
    '138': 'LaGuardia Airport',
    '161': 'Midtown Center',
    '162': 'Midtown East',
    '163': 'Midtown North',
    '164': 'Midtown South',
    '170': 'Murray Hill',
    '186': 'Penn Station/MSG',
    '234': 'Union Square',
    '48':  'Clinton East',
    '68':  'East Chelsea',
    '100': 'Garment District',
    '230': 'Times Square',
    '237': 'Upper East Side North',
    '236': 'Upper East Side South',
    '239': 'Upper West Side North',
    '238': 'Upper West Side South',
    '107': 'Gramercy',
    '113': 'Greenwich Village North',
    '114': 'Greenwich Village South',
    '261': 'World Trade Center',
    '209': 'SoHo',
    '87':  'East Harlem North',
    '88':  'East Harlem South',
}

# Buscar zonas conocidas en nuestros datos
print("Hotspots identificados y su cluster DBSCAN:")
print("=" * 80)
print(f"{'Zone ID':>8s} | {'Nombre':>25s} | {'Borough':>15s} | {'Cluster':>8s} | {'Viajes':>12s} | {'Tarifa':>8s}")
print("-" * 80)

for zone_id, name in sorted(known_zones.items(), key=lambda x: x[1]):
    row = df_clean[df_clean['location_id'] == zone_id]
    if len(row) > 0:
        r = row.iloc[0]
        cl = int(r['db_cluster'])
        cl_str = f'{cl}' if cl != -1 else 'Ruido'
        print(f"{zone_id:>8s} | {name:>25s} | {r['borough']:>15s} | {cl_str:>8s} | {r['trip_count']:>12,.0f} | ${r['avg_fare']:>6.2f}")
    else:
        print(f"{zone_id:>8s} | {name:>25s} | {'N/A':>15s} | {'N/A':>8s} | {'<1000':>12s} | {'N/A':>8s}")

print("\nNota: Zonas con menos de 1,000 viajes fueron filtradas del analisis.")

## Conclusiones

1. **Clustering por features de zona identifica patrones operativos reales:** Los clusters agrupan
   zonas con comportamientos similares en tarifa, propina, distancia y volumen de viajes,
   revelando diferencias operativas entre areas de NYC.

2. **DBSCAN vs K-Means:** DBSCAN es superior para este tipo de analisis porque:
   - Detecta zonas atipicas como ruido (ej: aeropuertos con tarifas fijas)
   - Descubre el numero natural de agrupaciones sin necesidad de especificarlo
   - No fuerza zonas muy diferentes a pertenecer a un cluster

3. **Diferencias economicas por cluster:** Los clusters revelan perfiles economicos distintos:
   zonas de aeropuerto tienen tarifas mas altas y distancias mayores, mientras que zonas
   centrales de Manhattan tienen viajes mas cortos pero con mayor volumen y propina porcentual.

4. **t-SNE y PCA:** La reduccion dimensional confirma que las features de zona (tarifa,
   distancia, volumen, propina) capturan diferencias reales entre grupos de zonas, y que
   la geografia (borough) esta correlacionada con los patrones de viaje.

### Proximos pasos
- Notebook 03: Descomposicion de series temporales de la demanda diaria